# Research 12 — Four-way ensemble after the loyalty × VIP screen

The LightGBM loyalty × VIP model improved the current blend by only 0.000279 macro F1 and had mixed fold deltas. **Hypothesis:** a 10% contribution may add useful signal to the previously strongest 80/10/10 ensemble without replacing its diverse XGBoost and balanced-CatBoost components. Test one fixed 70/10/10/10 mixture; no weight sweep or model fitting. All inputs are aligned training OOF predictions, and Test is not loaded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)
base_frame = pd.read_csv(ROOT / "predictions/segment-oof.csv")
assert len(base_frame) == len(y) and np.array_equal(base_frame.segment.to_numpy(), y)


def load(name):
    frame = pd.read_csv(ROOT / "predictions" / name)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), name
    values = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(values).all() and np.all(values >= 0)
    sums = values.sum(axis=1, keepdims=True)
    assert np.all(sums > 0)
    return values / sums


p_base = load("segment-oof.csv")
p_xgb = load("segment-xgboost-oof.csv")
p_balanced = load("segment-balanced-catboost-oof.csv")
p_vip = load("segment-lightgbm-loyalty-vip-oof.csv")
p_existing_best = 0.8 * p_base + 0.1 * p_xgb + 0.1 * p_balanced
p_vip_blend = 0.8 * p_base + 0.2 * p_vip
p_four_way = 0.7 * p_base + 0.1 * p_xgb + 0.1 * p_balanced + 0.1 * p_vip
candidates = {
    "Current CatBoost/LightGBM": p_base,
    "Existing 80/10/10 (XGB + balanced CatBoost)": p_existing_best,
    "80/20 current + loyalty × VIP": p_vip_blend,
    "70/10/10/10 four-way mixture": p_four_way,
}
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
base_folds = results.loc[
    results.candidate == "Current CatBoost/LightGBM",
    [f"fold_{fold}_macro_f1" for fold in range(1, 4)],
].iloc[0]
for fold in range(1, 4):
    results[f"fold_{fold}_delta"] = (
        results[f"fold_{fold}_macro_f1"] - base_folds[f"fold_{fold}_macro_f1"]
    )
display(results.round(6))
results.to_csv(ROOT / "assets/segment-four-way-synthesis.csv", index=False)
pd.DataFrame(p_four_way, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-four-way-mixture-oof.csv", index=False
)
print(
    "Probability rows were normalized before averaging. OOF-only screening; no Test or submission access."
)

,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1,fold_1_delta,fold_2_delta,fold_3_delta
0,Current CatBoost/LightGBM,0.750895,0.764683,0.749445,0.749821,0.753390,0.000000,0.000000,0.000000
1,Existing 80/10/10 (XGB + balanced CatBoost),0.751142,0.764884,0.749915,0.750579,0.752899,0.000470,0.000758,-0.000491
2,80/20 current + loyalty × VIP,0.751174,0.764949,0.749521,0.749686,0.754286,0.000076,-0.000135,0.000896
3,70/10/10/10 four-way mixture,0.751321,0.765050,0.749899,0.750745,0.753290,0.000454,0.000924,-0.000100


Probability rows were normalized before averaging. OOF-only screening; no Test or submission access.


## Decision

This single fixed combination tests whether the weakly positive loyalty × VIP candidate adds value to the existing ensemble. The three reused OOF folds are not an independent test, so small changes remain inconclusive.